# Task 5: Per-Tensor vs Per-Channel Symmetric Quantization

## Objective
Compare **per-tensor** and **per-output-channel** symmetric quantization for convolution weights and observe why per-channel quantization usually provides better accuracy.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

conv_weights = np.random.randn(8,3,3,3).astype(np.float32)

# Create range imbalance
conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

print("Weight tensor shape:", conv_weights.shape)


Weight tensor shape: (8, 3, 3, 3)


## Part A: Per-Tensor Symmetric Quantization

A single scale is computed for the entire tensor.

Formula:

- Scale = max(|tensor|) / 127
- Quantized = round(weight / scale)
- Dequantized = quantized × scale


In [2]:
def per_tensor_quantize(tensor):
    max_abs = np.max(np.abs(tensor))
    scale = max_abs/127 if max_abs!=0 else 1.0

    q = np.round(tensor/scale)
    q = np.clip(q,-127,127).astype(np.int8)

    dq = q.astype(np.float32)*scale

    return q,dq,scale


## Part B: Per-Output-Channel Symmetric Quantization

Each output channel (axis=0) receives its own scale.

Formula:

- scale[c] = max(|channel|) / 127


In [3]:
def per_channel_quantize(tensor):
    q=np.zeros_like(tensor,dtype=np.int8)
    dq=np.zeros_like(tensor,dtype=np.float32)
    scales=[]

    for c in range(tensor.shape[0]):
        channel=tensor[c]
        max_abs=np.max(np.abs(channel))
        scale=max_abs/127 if max_abs!=0 else 1.0
        scales.append(scale)

        qc=np.round(channel/scale)
        qc=np.clip(qc,-127,127).astype(np.int8)

        q[c]=qc
        dq[c]=qc.astype(np.float32)*scale

    return q,dq,np.array(scales)


## Part C: Quantize and Compare

In [4]:
pt_q,pt_dq,pt_scale = per_tensor_quantize(conv_weights)
pc_q,pc_dq,pc_scales = per_channel_quantize(conv_weights)

rows=[]

for c in range(conv_weights.shape[0]):
    w=conv_weights[c]

    pt_mae=np.mean(np.abs(w-pt_dq[c]))
    pc_mae=np.mean(np.abs(w-pc_dq[c]))

    better="Per-Channel" if pc_mae<pt_mae else "Per-Tensor"

    rows.append([
        c,
        f"({w.min():.3f}, {w.max():.3f})",
        pt_scale,
        pt_mae,
        pc_scales[c],
        pc_mae,
        better
    ])

avg_pt=np.mean([r[3] for r in rows])
avg_pc=np.mean([r[5] for r in rows])

rows.append([
    "Average",
    "-",
    pt_scale,
    avg_pt,
    np.mean(pc_scales),
    avg_pc,
    "Per-Channel" if avg_pc<avg_pt else "Per-Tensor"
])

df=pd.DataFrame(rows,columns=[
"Channel",
"Range (min,max)",
"Per-Tensor Scale",
"Per-Tensor MAE",
"Per-Ch Scale",
"Per-Ch MAE",
"Better"
])

print(df)


   Channel    Range (min,max)  Per-Tensor Scale  Per-Tensor MAE  Per-Ch Scale  \
0        0    (-0.191, 0.158)          0.303365        0.071464      0.001507   
1        1    (-0.980, 0.926)          0.303365        0.072564      0.007715   
2        2    (-2.620, 1.565)          0.303365        0.072175      0.020628   
3        3    (-1.464, 1.886)          0.303365        0.071594      0.014852   
4        4    (-1.919, 2.463)          0.303365        0.070571      0.019396   
5        5    (-1.607, 1.866)          0.303365        0.068939      0.014691   
6        6   (-5.354, 13.601)          0.303365        0.077800      0.107093   
7        7  (-15.148, 38.527)          0.303365        0.064038      0.303365   
8  Average                  -          0.303365        0.071143      0.061156   

   Per-Ch MAE       Better  
0    0.000326  Per-Channel  
1    0.001661  Per-Channel  
2    0.005373  Per-Channel  
3    0.003731  Per-Channel  
4    0.003995  Per-Channel  
5    0.003901  

## Part D: Example Reconstruction

Display the original, per-tensor dequantized and per-channel dequantized weights for one output channel.


In [5]:
channel=7

print("Original Channel")
print(conv_weights[channel])

print("\nPer-Tensor Dequantized")
print(pt_dq[channel])

print("\nPer-Channel Dequantized")
print(pc_dq[channel])


Original Channel
[[[-15.148473    -4.46515      8.563988  ]
  [  2.1409373  -12.457388     1.7318093 ]
  [  3.8531737   -8.838574     1.537251  ]]

 [[  0.58208716 -11.429703     3.5778737 ]
  [  5.6078453   10.830512    10.53802   ]
  [-13.776693    -9.37825      5.1503525 ]]

 [[  5.1378593    5.1504765   38.527313  ]
  [  5.7089047   11.355657     9.540018  ]
  [  6.5139127   -3.1526923    7.5896926 ]]]

Per-Tensor Dequantized
[[[-15.168233   -4.55047     8.49421  ]
  [  2.1235526 -12.437951    1.820188 ]
  [  3.9437406  -8.797575    1.5168233]]

 [[  0.6067293 -11.527857    3.640376 ]
  [  5.460564   10.921128   10.6177635]
  [-13.65141    -9.4043045   5.1571994]]

 [[  5.1571994   5.1571994  38.527313 ]
  [  5.7639284  11.224493    9.4043045]
  [  6.370658   -3.0336466   7.5841165]]]

Per-Channel Dequantized
[[[-15.168233   -4.55047     8.49421  ]
  [  2.1235526 -12.437951    1.820188 ]
  [  3.9437406  -8.797575    1.5168233]]

 [[  0.6067293 -11.527857    3.640376 ]
  [  5.460564

# Analysis Questions

## 1. Why does per-channel quantization usually produce lower error?

Per-channel quantization computes an independent scale for every output channel. Each channel therefore uses the full INT8 dynamic range, reducing rounding error and improving reconstruction accuracy.

---

## 2. Which channels benefit the most from per-channel quantization, and why?

Channels with very small ranges (such as Channel 0 and Channel 1) benefit the most because a global scale is dominated by the largest channels. Their values become much more accurately represented when they receive their own scale.

---

## 3. Why do channels with large value ranges show a smaller difference?

Large-range channels already determine the global scale in per-tensor quantization. Therefore, the global scale is already well suited for these channels, so per-channel quantization provides only a small additional improvement.

---

## 4. What is the main trade-off?

### Per-Tensor
- Simpler implementation
- Only one scale stored
- Faster inference
- Lower memory overhead
- Higher quantization error when channel ranges differ significantly

### Per-Channel
- Lower quantization error
- Better model accuracy
- One scale stored per output channel
- Slightly higher memory usage and implementation complexity
